# 22 训练Transformer - 从零训练完成实际任务

本教程将带你从零开始训练一个Transformer模型，完成一个**序列复制任务**（学习复制输入序列），帮助你理解训练的每个细节。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import time

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("导入库成功！")

## 一、训练任务设计

### 任务：序列复制（Copy Task）

让Transformer学习**复制输入序列**——这是一个简单但完整的Seq2Seq任务，能验证Transformer的所有核心机制。

```
输入: [3, 7, 2, 5, 1]
输出: [3, 7, 2, 5, 1]
```

为什么选这个任务？
1. **数据简单可控**——不需要下载任何数据集
2. **验证所有机制**——模型必须学会：位置编码、自注意力、交叉注意力、自回归生成
3. **容易调试**——你知道正确答案是什么
4. **快速训练**——在CPU上几分钟就能训练完成

In [ ]:
# 生成训练数据
print("=" * 70)
print("生成Copy Task数据")
print("=" * 70)
print()

# 特殊token
PAD_TOKEN = 0
SOS_TOKEN = 1
EOS_TOKEN = 2
VOCAB_SIZE = 20  # 0=PAD, 1=SOS, 2=EOS, 3~19=实际词

def generate_copy_data(n_samples, seq_len):
    """
    生成复制任务数据
    输入: [SOS, a, b, c, ..., EOS, PAD, PAD]
    目标: [SOS, a, b, c, ..., EOS, PAD, PAD]
    """
    max_len = seq_len + 2  # +SOS +EOS
    
    src = torch.full((n_samples, max_len), PAD_TOKEN, dtype=torch.long)
    tgt = torch.full((n_samples, max_len), PAD_TOKEN, dtype=torch.long)
    
    for i in range(n_samples):
        # 随机序列长度 (3~seq_len)
        length = torch.randint(3, seq_len + 1, (1,)).item()
        sequence = torch.randint(3, VOCAB_SIZE, (length,))
        
        # 源序列: SOS + 序列 + EOS
        src[i, 0] = SOS_TOKEN
        src[i, 1:length+1] = sequence
        src[i, length+1] = EOS_TOKEN
        
        # 目标序列（相同）
        tgt[i, 0] = SOS_TOKEN
        tgt[i, 1:length+1] = sequence
        tgt[i, length+1] = EOS_TOKEN
    
    return src, tgt

# 生成数据
seq_len = 8
n_train = 2000
n_test = 200

src_train, tgt_train = generate_copy_data(n_train, seq_len)
src_test, tgt_test = generate_copy_data(n_test, seq_len)

print(f"训练集: src={src_train.shape}, tgt={tgt_train.shape}")
print(f"测试集: src={src_test.shape}, tgt={tgt_test.shape}")
print()
print(f"特殊Token: PAD={PAD_TOKEN}, SOS={SOS_TOKEN}, EOS={EOS_TOKEN}")
print(f"词汇表大小: {VOCAB_SIZE}")
print()
print("训练数据示例:")
for i in range(3):
    print(f"  样本{i}: src={src_train[i].tolist()}")
    print(f"           tgt={tgt_train[i].tolist()}")
    print()

## 二、构建训练用的Transformer模型

我们使用一个**小型Transformer**，适合CPU训练。

In [ ]:
# 定义小型Transformer模型
print("=" * 70)
print("构建小型Transformer模型")
print("=" * 70)
print()

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:x.size(1), :].unsqueeze(0)

class SmallTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, 
                 d_ff=128, max_len=50, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # 编码器和解码器（复制任务两者结构相同）
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True  # Pre-Norm
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        
        self.src_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_TOKEN)
        self.tgt_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_TOKEN)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, 
                src_padding_mask=None, tgt_padding_mask=None, memory_mask=None):
        # 编码
        src_embedded = self.dropout(self.pos_encoding(self.src_embedding(src)))
        memory = self.encoder(src_embedded, mask=src_mask, src_key_padding_mask=src_padding_mask)
        
        # 解码
        tgt_embedded = self.dropout(self.pos_encoding(self.tgt_embedding(tgt)))
        output = self.decoder(tgt_embedded, memory, tgt_mask=tgt_mask,
                            memory_mask=memory_mask,
                            tgt_key_padding_mask=tgt_padding_mask,
                            memory_key_padding_mask=src_padding_mask)
        
        return self.fc_out(output)

# 创建模型
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 128

model = SmallTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    max_len=seq_len + 5
)

print(f"模型配置:")
print(f"  d_model = {d_model}")
print(f"  n_heads = {n_heads}")
print(f"  n_layers = {n_layers}")
print(f"  d_ff = {d_ff}")
print(f"  vocab_size = {VOCAB_SIZE}")
print()
print(f"总参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"  嵌入层: {sum(p.numel() for p in model.src_embedding.parameters()) + sum(p.numel() for p in model.tgt_embedding.parameters()):,}")
print(f"  编码器: {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"  解码器: {sum(p.numel() for p in model.decoder.parameters()):,}")
print(f"  输出层: {sum(p.numel() for p in model.fc_out.parameters()):,}")

## 三、训练循环

### 关键训练技巧

1. **Teacher Forcing**：训练时解码器的输入是真实标签（而非自己的预测）
2. **Label Smoothing**：让目标概率分布稍微平滑一些，防止过拟合
3. **梯度裁剪**：防止梯度爆炸
4. **Warmup学习率**：学习率先线性增长再衰减

In [ ]:
# 训练Transformer
print("=" * 70)
print("训练Transformer")
print("=" * 70)
print()

torch.manual_seed(42)
model = SmallTransformer(
    vocab_size=VOCAB_SIZE, d_model=64, n_heads=4, n_layers=2, d_ff=128,
    max_len=seq_len + 5
)

# 损失函数：带label smoothing的交叉熵
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN, label_smoothing=0.1)

# 优化器：Adam with warmup
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.98), eps=1e-9)

# Warmup学习率调度器
warmup_steps = 200
def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps  # 线性warmup
    else:
        return max(0.1, 0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / 500)))  # 余弦衰减

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# 训练参数
batch_size = 32
n_epochs = 50

# 创建mask
def create_masks(src, tgt):
    src_pad_mask = (src == PAD_TOKEN)
    tgt_pad_mask = (tgt == PAD_TOKEN)
    
    # Decoder的自回归mask（下三角）
    tgt_len = tgt.size(1)
    tgt_mask = torch.tril(torch.ones(tgt_len, tgt_len, dtype=torch.bool))
    
    return src_pad_mask, tgt_pad_mask, tgt_mask

# 训练
loss_history = []
acc_history = []
lr_history = []
step_count = 0

print(f"{'Epoch':<6} {'Train Loss':>12} {'Train Acc':>10} {'LR':>10}")
print("-" * 42)

start_time = time.time()

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    epoch_correct = 0
    epoch_total = 0
    n_batches = 0
    
    # Mini-batch训练
    indices = torch.randperm(n_train)
    for start in range(0, n_train, batch_size):
        end = min(start + batch_size, n_train)
        batch_idx = indices[start:end]
        
        src_batch = src_train[batch_idx]
        tgt_batch = tgt_train[batch_idx]
        
        src_pad_mask, tgt_pad_mask, tgt_mask = create_masks(src_batch, tgt_batch)
        
        # 前向传播
        output = model(
            src_batch, tgt_batch,
            tgt_mask=tgt_mask,
            src_padding_mask=src_pad_mask,
            tgt_padding_mask=tgt_pad_mask
        )
        
        # 计算损失（忽略PAD和SOS位置）
        output_flat = output[:, 1:].reshape(-1, VOCAB_SIZE)  # 从第2个token开始
        tgt_flat = tgt_batch[:, 1:].reshape(-1)
        
        loss = criterion(output_flat, tgt_flat)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        epoch_loss += loss.item()
        n_batches += 1
        step_count += 1
        
        # 计算准确率（非PAD位置）
        non_pad_mask = (tgt_flat != PAD_TOKEN)
        if non_pad_mask.any():
            preds = output_flat[non_pad_mask].argmax(dim=-1)
            epoch_correct += (preds == tgt_flat[non_pad_mask]).sum().item()
            epoch_total += non_pad_mask.sum().item()
    
    avg_loss = epoch_loss / n_batches
    accuracy = epoch_correct / epoch_total if epoch_total > 0 else 0
    current_lr = optimizer.param_groups[0]['lr']
    
    loss_history.append(avg_loss)
    acc_history.append(accuracy)
    lr_history.append(current_lr)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"{epoch+1:<6} {avg_loss:>12.4f} {accuracy:>10.2%} {current_lr:>10.6f}")

train_time = time.time() - start_time
print(f"\n训练完成！用时: {train_time:.1f}s")
print(f"最终训练损失: {loss_history[-1]:.4f}")
print(f"最终训练准确率: {acc_history[-1]:.2%}")

In [ ]:
# 可视化训练结果
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 损失曲线
axes[0].plot(loss_history, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title(f'Training Loss\n(最终: {loss_history[-1]:.4f})')
axes[0].grid(True, alpha=0.3)

# 准确率曲线
axes[1].plot(acc_history, 'g-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title(f'Training Accuracy\n(最终: {acc_history[-1]:.2%})')
axes[1].grid(True, alpha=0.3)

# 学习率曲线
axes[2].plot(lr_history, 'r-', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule\n(Warmup + Cosine Decay)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 四、推理（自回归生成）

训练完成后，我们需要用模型进行**推理**——逐步生成输出序列。

In [ ]:
# 推理：自回归生成
print("=" * 70)
print("Transformer推理 - 自回归生成")
print("=" * 70)
print()

def greedy_decode(model, src, max_len):
    """
    贪婪解码：每次选择概率最高的token
    """
    model.eval()
    
    # 编码
    src_pad_mask = (src == PAD_TOKEN)
    with torch.no_grad():
        src_embedded = model.pos_encoding(model.src_embedding(src))
        memory = model.encoder(src_embedded, src_key_padding_mask=src_pad_mask)
    
    # 解码：逐步生成
    batch_size = src.size(0)
    tgt = torch.full((batch_size, 1), SOS_TOKEN, dtype=torch.long)
    
    for i in range(max_len - 1):
        tgt_pad_mask = (tgt == PAD_TOKEN)
        tgt_mask = torch.tril(torch.ones(tgt.size(1), tgt.size(1), dtype=torch.bool))
        
        with torch.no_grad():
            tgt_embedded = model.pos_encoding(model.tgt_embedding(tgt))
            output = model.decoder(tgt_embedded, memory,
                                  tgt_mask=tgt_mask,
                                  tgt_key_padding_mask=tgt_pad_mask,
                                  memory_key_padding_mask=src_pad_mask)
            logits = model.fc_out(output[:, -1, :])  # 只取最后一个位置
            next_token = logits.argmax(dim=-1, keepdim=True)
        
        tgt = torch.cat([tgt, next_token], dim=1)
        
        # 如果所有样本都生成了EOS，提前结束
        if (next_token == EOS_TOKEN).all():
            break
    
    return tgt

# 测试推理
n_test_display = 5
model.eval()

print(f"{'#':<4} {'输入序列':<35} {'预测输出':<35} {'是否正确':>10}")
print("-" * 90)

correct_count = 0
for i in range(n_test_display):
    src_sample = src_test[i:i+1]
    tgt_sample = tgt_test[i:i+1]
    
    # 推理
    pred = greedy_decode(model, src_sample, seq_len + 5)
    
    # 转换为可读格式
    src_seq = src_sample[0].tolist()
    tgt_seq = tgt_sample[0].tolist()
    pred_seq = pred[0].tolist()
    
    # 去掉PAD
    src_clean = [t for t in src_seq if t != PAD_TOKEN]
    tgt_clean = [t for t in tgt_seq if t != PAD_TOKEN]
    pred_clean = [t for t in pred_seq if t != PAD_TOKEN]
    
    is_correct = (pred_clean == tgt_clean)
    if is_correct:
        correct_count += 1
    
    src_str = str(src_clean)[:30]
    pred_str = str(pred_clean)[:30]
    
    print(f"{i+1:<4} {src_str:<35} {pred_str:<35} {'✓' if is_correct else '✗':>10}")

print()
print(f"前{n_test_display}个样本的推理正确率: {correct_count}/{n_test_display} = {correct_count/n_test_display:.0%}")

In [ ]:
# 批量评估
print("\n=== 批量评估 ===")
print()

model.eval()
total_correct = 0
total = n_test

for i in range(n_test):
    src_sample = src_test[i:i+1]
    tgt_sample = tgt_test[i:i+1]
    
    pred = greedy_decode(model, src_sample, seq_len + 5)
    
    # 比较（忽略PAD和EOS之后的内容）
    tgt_clean = [t for t in tgt_sample[0].tolist() if t != PAD_TOKEN]
    pred_clean = [t for t in pred[0].tolist() if t != PAD_TOKEN]
    
    if pred_clean == tgt_clean:
        total_correct += 1

print(f"测试集准确率: {total_correct}/{total} = {total_correct/total:.2%}")

# 分析错误
print("\n错误分析:")
for i in range(n_test):
    src_sample = src_test[i:i+1]
    tgt_sample = tgt_test[i:i+1]
    pred = greedy_decode(model, src_sample, seq_len + 5)
    
    tgt_clean = [t for t in tgt_sample[0].tolist() if t != PAD_TOKEN]
    pred_clean = [t for t in pred[0].tolist() if t != PAD_TOKEN]
    
    if pred_clean != tgt_clean and i < 10:  # 只显示前10个错误
        print(f"  样本{i}: 输入={src_sample[0].tolist()}")
        print(f"           期望={tgt_clean}")
        print(f"           预测={pred_clean}")

## 五、注意力权重可视化

让我们观察训练后的Transformer学到了什么样的注意力模式。

In [ ]:
# 提取并可视化注意力权重
print("\n=== 注意力权重可视化 ===")
print()

# 创建一个能返回注意力权重的模型
class TransformerWithAttention(SmallTransformer):
    def encode(self, src, src_pad_mask):
        src_embedded = self.dropout(self.pos_encoding(self.src_embedding(src)))
        
        x = src_embedded
        enc_attns = []
        for layer in self.encoder.layers:
            # 手动执行层以获取注意力
            x2 = layer.norm1(x)
            x2, attn = layer.self_attn(x2, x2, x2,
                                        key_padding_mask=src_pad_mask,
                                        need_weights=True,
                                        is_causal=False)
            x = x + layer.dropout1(x2)
            x2 = layer.norm2(x)
            x2 = layer.linear2(layer.dropout(layer.activation(layer.linear1(x2))))
            x = x + layer.dropout2(x2)
            enc_attns.append(attn)
        
        return x, enc_attns

# 用一个简单样本测试注意力
model_attn = TransformerWithAttention(
    vocab_size=VOCAB_SIZE, d_model=64, n_heads=4, n_layers=2, d_ff=128,
    max_len=seq_len + 5
)
model_attn.load_state_dict(model.state_dict())
model_attn.eval()

# 简单测试样本
test_src = src_test[0:1]
test_tgt = tgt_test[0:1]

src_pad_mask = (test_src == PAD_TOKEN)

with torch.no_grad():
    _, enc_attns = model_attn.encode(test_src, src_pad_mask)

print(f"编码器层数: {len(enc_attns)}")
print(f"每层注意力形状: {enc_attns[0].shape}")
print()

# 可视化注意力
n_layers = len(enc_attns)
n_heads = enc_attns[0].shape[0]

fig, axes = plt.subplots(n_layers, n_heads, figsize=(4*n_heads, 3*n_layers))
if n_layers == 1:
    axes = axes.reshape(1, -1)

# 获取token对应的词
src_tokens = test_src[0].tolist()
src_words = [str(t) for t in src_tokens if t != PAD_TOKEN]
actual_len = len(src_words)

for layer_idx in range(n_layers):
    for head_idx in range(n_heads):
        ax = axes[layer_idx, head_idx]
        attn = enc_attns[layer_idx][head_idx].detach().numpy()
        attn = attn[:actual_len, :actual_len]
        
        im = ax.imshow(attn, cmap='Blues', interpolation='nearest')
        ax.set_xticks(range(actual_len))
        ax.set_yticks(range(actual_len))
        ax.set_xticklabels(src_words, fontsize=8, rotation=45)
        ax.set_yticklabels(src_words, fontsize=8)
        
        if layer_idx == 0:
            ax.set_title(f'Head {head_idx+1}', fontsize=10)
        if head_idx == 0:
            ax.set_ylabel(f'Layer {layer_idx+1}', fontsize=10)
        
        # 标数值
        for i in range(actual_len):
            for j in range(actual_len):
                val = attn[i, j]
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                       fontsize=6, color='white' if val > 0.5 else 'black')

plt.suptitle('Encoder Self-Attention Weights', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("注意力权重解读:")
print("  - 每行和为1（softmax归一化）")
print("  - 对角线权重高 = 关注自己")
print("  - 某行关注特定列 = 学习到特定关系")
print("  - 均匀分布 = 还没学到有用的信息")

## 六、本章总结

### 训练流程回顾

1. **数据准备**：生成Copy Task的源/目标序列对
2. **模型构建**：小型Transformer (d_model=64, 2层, 4头)
3. **训练**：
   - Teacher Forcing（用真实标签作为解码器输入）
   - Adam优化器 + Warmup学习率
   - 梯度裁剪 + Label Smoothing
4. **推理**：自回归贪婪解码
5. **评估**：检查生成序列是否正确
6. **注意力可视化**：理解模型学到的模式

### 训练关键要点

| 技巧 | 作用 |
|------|------|
| Teacher Forcing | 加速训练，防止误差累积 |
| Label Smoothing | 防止过拟合，提升泛化 |
| Gradient Clipping | 防止梯度爆炸 |
| Warmup LR | 稳定训练初期，避免大梯度 |
| Pre-Norm | 让深层训练更稳定 |

### 下一篇预告

下一篇将学习**Transformer的高级优化技术**：RoPE、Flash Attention、KV Cache、GQA、SwiGLU等现代大模型使用的技术。